# MongoDB Fundamentals

## Notebook Objectives

This notebook introduces the fundamentals of MongoDB using both **PyMongo** and **MongoDB Shell (mongosh)**.

Throughout this notebook, every MongoDB operation will be demonstrated using:

- **PyMongo** – The official Python driver for MongoDB.
- **MongoDB Shell (mongosh)** – The command-line interface used to interact with MongoDB.

By the end of this notebook, you will be able to:

- Connect to a MongoDB server.
- Create databases and collections.
- Insert, retrieve, update, and delete documents.
- Use MongoDB query operators.
- Perform aggregation pipeline operations.

> **Note:** For operations that modify the database (Insert, Update, Delete), the **mongosh** commands are provided as commented code to prevent duplicate execution. You can copy and execute them directly in the MongoDB Shell whenever required.

# Import Required Libraries

The following Python libraries will be used throughout this notebook.

| Library | Purpose |
|----------|----------|
| `pymongo` | Connect Python applications with MongoDB |
| `pandas` | Data manipulation and analysis |
| `numpy` | Numerical computations |
| `matplotlib` | Data visualization |
| `seaborn` | Statistical visualization |

In [2]:
# =====================================
# Import Required Libraries
# =====================================

from pymongo import MongoClient
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint

print("All required libraries imported successfully.")

All required libraries imported successfully.


# Connect to MongoDB Server

Before performing any database operations, we must establish a connection with the MongoDB server.

The default connection URL is:

```text
mongodb://localhost:27017
```

where

- **localhost** → MongoDB Server
- **27017** → Default MongoDB Port

After creating the client, we will verify that the MongoDB server is running successfully.

In [3]:
# =====================================
# PyMongo Command
# =====================================

client = MongoClient("mongodb://localhost:27017")

try:
    client.admin.command("ping")
    print("Successfully connected to MongoDB Server.")
except Exception as e:
    print("Connection Failed")
    print(e)

Successfully connected to MongoDB Server.


In [5]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.adminCommand({ ping: 1 })"

{ ok: 1 }


# Display Available Databases

MongoDB stores multiple databases on a single server.

The following commands display all currently available databases.

> **Note:** MongoDB displays only those databases that contain at least one collection with data.

In [9]:
# =====================================
# PyMongo Command
# =====================================

client.list_database_names()

['admin', 'config', 'local']

In [10]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "show dbs"

admin    40.00 KiB
config  108.00 KiB
local    64.00 KiB


# Database

A **database** is a logical container that stores one or more **collections**.

Just as a folder on your computer can contain multiple files, a MongoDB database can contain multiple collections.

In this notebook, we will create a database named **ecommerce**.

> **Important:**  
> In MongoDB, simply selecting a database **does not create it**. The database is physically created **only after a document is inserted into one of its collections**.

For now, we will simply select the database. It will appear in MongoDB only after data is inserted in later sections.

In [6]:
# =====================================
# PyMongo Command
# =====================================

db = client["ecommerce"]

db

Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'ecommerce')

In [12]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce"

switched to db ecommerce


# Understanding the Database Object

The variable **`db`** is a **Database object** returned by PyMongo.

It represents the selected MongoDB database and allows us to perform operations such as:

- Creating collections
- Listing collections
- Inserting documents
- Querying data
- Updating documents
- Deleting documents

Since no documents have been inserted yet, the **ecommerce** database does not physically exist on the MongoDB server.

You can verify this by listing all available databases.

In [13]:
# =====================================
# PyMongo Command
# =====================================

client.list_database_names()

['admin', 'config', 'local']

In [14]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "show dbs"

admin    40.00 KiB
config  108.00 KiB
local    64.00 KiB


# Collection

A **collection** is a group of related documents stored within a MongoDB database.

It is similar to a **table** in a relational database (RDBMS), but unlike a table:

- Collections do **not** require a fixed schema.
- Documents in the same collection can have different fields.
- MongoDB automatically creates a collection when the first document is inserted into it.

For better understanding, consider the following comparison:

| Relational Database | MongoDB |
|---------------------|----------|
| Database | Database |
| Table | Collection |
| Row | Document |
| Column | Field |

In this notebook, we will use a collection named **customers** to store customer information for our e-commerce application.

> **Important:**  
> Like databases, selecting a collection does **not** physically create it. The collection is created automatically when the first document is inserted.

In [7]:
# =====================================
# PyMongo Command
# =====================================

customers = db["customers"]

customers

Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'ecommerce'), 'customers')

In [16]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce; db.getCollection('customers')"

switched to db ecommerce;


# Understanding the Collection Object

The variable **customers** is a **Collection object**.

It represents the **customers** collection inside the **ecommerce** database.

Using this object, we can perform operations such as:

- Insert documents
- Retrieve documents
- Update documents
- Delete documents
- Aggregate data

Since no documents have been inserted yet, the **customers** collection has not been created physically.

In [17]:
# =====================================
# PyMongo Command
# =====================================

db.list_collection_names()

[]

In [18]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce; show collections"

switched to db ecommerce;


# Document

A **document** is the basic unit of data storage in MongoDB.

A document is a collection of **field-value pairs**, similar to a JSON object.

For example, a customer in an e-commerce application can be represented as a single document.

```json
{
    "customer_id": 1001,
    "name": "Rahul Sharma",
    "gender": "Male",
    "age": 28,
    "email": "rahul@example.com",
    "city": "Hyderabad",
    "membership": "Gold",
    "is_active": true
}
```

Each field stores a specific piece of information about the customer.

Unlike relational databases, MongoDB documents can have different fields. This flexible schema makes MongoDB suitable for applications where the data structure changes frequently.

# JSON vs BSON

Although MongoDB documents look like **JSON**, MongoDB actually stores them internally as **BSON (Binary JSON)**.

BSON extends JSON by supporting additional data types and enabling faster storage and retrieval.

| JSON | BSON |
|------|------|
| Text-based format | Binary format |
| Human-readable | Machine-efficient |
| Limited data types | Supports additional data types such as ObjectId, Date, Binary, Decimal128, etc. |
| Used for data interchange | Used for internal storage by MongoDB |

When we write documents in Python or in the MongoDB Shell, they appear like JSON, but MongoDB automatically converts them into BSON before storing them.

# The `_id` Field and ObjectId

Every document stored in MongoDB must contain a unique field named **`_id`**.

If we do not provide an `_id`, MongoDB automatically generates one.

The automatically generated `_id` is usually of type **ObjectId**.

Example:

```text
ObjectId("6878fd5f5d5d4e8c2f7c3b9a")
```

The `_id` field serves as the **primary key** of a MongoDB document.

Its purpose is to uniquely identify every document within a collection.

> **Note:**  
> During our first insert operation, we will not specify the `_id` field. MongoDB will generate it automatically.

In [19]:
# =====================================
# PyMongo Command
# =====================================

from bson import ObjectId

# Generate a sample ObjectId (for demonstration)

sample_id = ObjectId()

sample_id

ObjectId('6a5cb76683106d9367c89b86')

In [20]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "ObjectId()"

ObjectId('6a5cb79992d204f8f4d7a6c1')


# Insert a Single Document

The **`insert_one()`** method is used to insert a single document into a MongoDB collection.

If the specified database or collection does not already exist, MongoDB automatically creates them when the first document is inserted.

Since we have not yet inserted any data, this operation will:

- Create the **ecommerce** database.
- Create the **customers** collection.
- Insert one customer document.
- Automatically generate an **`_id`** field because we are not providing one.

In [21]:
# =====================================
# PyMongo Command
# =====================================

customer = {
    "customer_id": 1001,
    "name": "Rahul Sharma",
    "gender": "Male",
    "age": 28,
    "email": "rahul.sharma@example.com",
    "phone": "9876543210",
    "city": "Hyderabad",
    "state": "Telangana",
    "membership": "Gold",
    "join_date": "2025-01-15",
    "is_active": True
}

result = customers.insert_one(customer)

print("Inserted Document ID:", result.inserted_id)

Inserted Document ID: 6a5cb7de83106d9367c89b87


In [22]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval '
# use ecommerce
# db.customers.insertOne({
#     customer_id: 1001,
#     name: "Rahul Sharma",
#     gender: "Male",
#     age: 28,
#     email: "rahul.sharma@example.com",
#     phone: "9876543210",
#     city: "Hyderabad",
#     state: "Telangana",
#     membership: "Gold",
#     join_date: "2025-01-15",
#     is_active: true
# })'

# Understanding the Return Value

The `insert_one()` method returns an **InsertOneResult** object.

The most commonly used attribute is:

- **`inserted_id`** – Returns the `_id` of the newly inserted document.

This value can be stored and used later to retrieve, update, or delete the same document.

In [23]:
# =====================================
# PyMongo Command
# =====================================

type(result)

pymongo.results.InsertOneResult

In [24]:
# =====================================
# PyMongo Command
# =====================================

result.inserted_id

ObjectId('6a5cb7de83106d9367c89b87')

# Verify Database and Collection Creation

Earlier, the **ecommerce** database and **customers** collection did not physically exist.

After inserting the first document, MongoDB automatically creates both.

Let's verify this.

In [25]:
# =====================================
# PyMongo Command
# =====================================

client.list_database_names()

['admin', 'config', 'ecommerce', 'local']

In [26]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "show dbs"

admin       40.00 KiB
config     108.00 KiB
ecommerce   40.00 KiB
local       64.00 KiB


In [27]:
# =====================================
# PyMongo Command
# =====================================

db.list_collection_names()

['customers']

In [28]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce; show collections"

switched to db ecommerce;


# Retrieve a Single Document

The **`find_one()`** method retrieves the **first document** that matches the specified query.

If no query is provided, MongoDB returns the **first document** in the collection.

The returned value is a Python dictionary (`dict`) representing a MongoDB document.

In [29]:
# =====================================
# PyMongo Command
# =====================================

customers.find_one()

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'customer_id': 1001,
 'name': 'Rahul Sharma',
 'gender': 'Male',
 'age': 28,
 'email': 'rahul.sharma@example.com',
 'phone': '9876543210',
 'city': 'Hyderabad',
 'state': 'Telangana',
 'membership': 'Gold',
 'join_date': '2025-01-15',
 'is_active': True}

In [30]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce; db.customers.findOne()"

switched to db ecommerce;


# Retrieve All Documents

The **`find()`** method retrieves **all documents** that match the specified query.

If no query is provided, every document in the collection is returned.

Unlike `find_one()`, the `find()` method returns a **Cursor object**.

A cursor is an iterator that allows MongoDB to efficiently return documents one at a time instead of loading all documents into memory.

In [31]:
# =====================================
# PyMongo Command
# =====================================

customers.find()

In [38]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce; db.customers.find()"

switched to db ecommerce;


# Display Documents Using a Loop

Since the `find()` method returns a **Cursor**, we usually iterate through it using a loop.

Each iteration returns one document from the collection.

In [34]:
# =====================================
# PyMongo Command
# =====================================

for customer in customers.find():
    print(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'), 'customer_id': 1001, 'name': 'Rahul Sharma', 'gender': 'Male', 'age': 28, 'email': 'rahul.sharma@example.com', 'phone': '9876543210', 'city': 'Hyderabad', 'state': 'Telangana', 'membership': 'Gold', 'join_date': '2025-01-15', 'is_active': True}


In [39]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find()"

[
  {
    _id: ObjectId('6a5cb7de83106d9367c89b87'),
    customer_id: 1001,
    name: 'Rahul Sharma',
    gender: 'Male',
    age: 28,
    email: 'rahul.sharma@example.com',
    phone: '9876543210',
    city: 'Hyderabad',
    state: 'Telangana',
    membership: 'Gold',
    join_date: '2025-01-15',
    is_active: true
  }
]


# Count the Number of Documents

The **`count_documents()`** method returns the number of documents that match a specified filter.

Passing an empty filter `{}` counts every document in the collection.

In [40]:
# =====================================
# PyMongo Command
# =====================================

customers.count_documents({})

1

In [41]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.countDocuments({})"

1


# Insert Multiple Documents

The **`insert_many()`** method inserts multiple documents into a collection in a single operation.

This method accepts a **list of dictionaries**, where each dictionary represents one document.

Advantages of using `insert_many()`:

- Inserts multiple documents efficiently.
- Reduces communication with the MongoDB server.
- Returns the `_id` values of all inserted documents.

In this section, we will populate our **customers** collection with additional customer records for future examples.

In [42]:
# =====================================
# PyMongo Command
# =====================================

customers_list = [
    {
        "customer_id": 1002,
        "name": "Priya Reddy",
        "gender": "Female",
        "age": 25,
        "email": "priya.reddy@example.com",
        "phone": "9876543211",
        "city": "Hyderabad",
        "state": "Telangana",
        "membership": "Silver",
        "join_date": "2025-02-10",
        "is_active": True
    },
    {
        "customer_id": 1003,
        "name": "Arjun Kumar",
        "gender": "Male",
        "age": 32,
        "email": "arjun.kumar@example.com",
        "phone": "9876543212",
        "city": "Bengaluru",
        "state": "Karnataka",
        "membership": "Gold",
        "join_date": "2024-12-20",
        "is_active": True
    },
    {
        "customer_id": 1004,
        "name": "Sneha Patel",
        "gender": "Female",
        "age": 29,
        "email": "sneha.patel@example.com",
        "phone": "9876543213",
        "city": "Mumbai",
        "state": "Maharashtra",
        "membership": "Platinum",
        "join_date": "2025-03-05",
        "is_active": True
    },
    {
        "customer_id": 1005,
        "name": "Vikram Singh",
        "gender": "Male",
        "age": 35,
        "email": "vikram.singh@example.com",
        "phone": "9876543214",
        "city": "Chennai",
        "state": "Tamil Nadu",
        "membership": "Silver",
        "join_date": "2024-11-18",
        "is_active": False
    },
    {
        "customer_id": 1006,
        "name": "Ananya Das",
        "gender": "Female",
        "age": 27,
        "email": "ananya.das@example.com",
        "phone": "9876543215",
        "city": "Kolkata",
        "state": "West Bengal",
        "membership": "Gold",
        "join_date": "2025-01-28",
        "is_active": True
    },
    {
        "customer_id": 1007,
        "name": "Rohit Verma",
        "gender": "Male",
        "age": 31,
        "email": "rohit.verma@example.com",
        "phone": "9876543216",
        "city": "Pune",
        "state": "Maharashtra",
        "membership": "Bronze",
        "join_date": "2025-04-01",
        "is_active": True
    },
    {
        "customer_id": 1008,
        "name": "Kavya Nair",
        "gender": "Female",
        "age": 26,
        "email": "kavya.nair@example.com",
        "phone": "9876543217",
        "city": "Kochi",
        "state": "Kerala",
        "membership": "Silver",
        "join_date": "2025-02-15",
        "is_active": False
    },
    {
        "customer_id": 1009,
        "name": "Amit Joshi",
        "gender": "Male",
        "age": 38,
        "email": "amit.joshi@example.com",
        "phone": "9876543218",
        "city": "Jaipur",
        "state": "Rajasthan",
        "membership": "Gold",
        "join_date": "2024-10-12",
        "is_active": True
    },
    {
        "customer_id": 1010,
        "name": "Neha Sharma",
        "gender": "Female",
        "age": 24,
        "email": "neha.sharma@example.com",
        "phone": "9876543219",
        "city": "Delhi",
        "state": "Delhi",
        "membership": "Bronze",
        "join_date": "2025-05-08",
        "is_active": True
    },
    {
        "customer_id": 1011,
        "name": "Suresh Iyer",
        "gender": "Male",
        "age": 40,
        "email": "suresh.iyer@example.com",
        "phone": "9876543220",
        "city": "Chennai",
        "state": "Tamil Nadu",
        "membership": "Platinum",
        "join_date": "2024-09-25",
        "is_active": True
    }
]

result = customers.insert_many(customers_list)

print("Inserted", len(result.inserted_ids), "documents.")

Inserted 10 documents.


In [43]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "
# db.getSiblingDB('ecommerce').customers.insertMany([
#     { customer_id:1002, name:'Priya Reddy', gender:'Female', age:25, email:'priya.reddy@example.com', phone:'9876543211', city:'Hyderabad', state:'Telangana', membership:'Silver', join_date:'2025-02-10', is_active:true },
#     { customer_id:1003, name:'Arjun Kumar', gender:'Male', age:32, email:'arjun.kumar@example.com', phone:'9876543212', city:'Bengaluru', state:'Karnataka', membership:'Gold', join_date:'2024-12-20', is_active:true }
#     // Remaining documents omitted for readability.
# ])
# "

# Verify the Inserted Documents

After inserting multiple documents, let's verify the total number of documents available in the **customers** collection.

In [44]:
# =====================================
# PyMongo Command
# =====================================

customers.count_documents({})

11

In [45]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.countDocuments({})"

11


In [55]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"membership": "Gold"})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id': 1006,
 'email': 'ananya.das@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-01-28',
 'membership': 'Gold',
 'name': 'Ananya Das',
 'phone': '9876543215',
 'state': 'West Bengal'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009,

In [56]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({membership:'Gold'}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,
  email: 'ananya.das@example.com',
  phone: '9876543215',
  city: 'Kolkata',
  state: 'West Bengal',
  membership: 'Gold',
  join_date: '2025-01-28',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  emai

# Query Documents Using Filters

In the previous section, we retrieved all documents from the collection.

However, in real-world applications, we rarely need every document. Instead, we retrieve only those documents that satisfy a specific condition.

MongoDB uses a **query filter** to specify the search criteria.

A query filter is written as a Python dictionary (or a JSON document in the MongoDB Shell).

General Syntax:

```python
collection.find(filter)
```

If the filter is empty (`{}`), all documents are returned.

If the filter contains conditions, only the matching documents are returned.

# Equality Query

The simplest type of query is an **equality query**.

An equality query returns all documents where the specified field exactly matches the given value.

For example, the following query retrieves all customers whose city is **Hyderabad**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE city = 'Hyderabad';
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"city": "Hyderabad"})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({city:'Hyderabad'}).pretty()"

[
  {
    _id: ObjectId('6a5cb7de83106d9367c89b87'),
    customer_id: 1001,
    name: 'Rahul Sharma',
    gender: 'Male',
    age: 28,
    email: 'rahul.sharma@example.com',
    phone: '9876543210',
    city: 'Hyderabad',
    state: 'Telangana',
    membership: 'Gold',
    join_date: '2025-01-15',
    is_active: true
  },
  {
    _id: ObjectId('6a5cba4483106d9367c89b88'),
    customer_id: 1002,
    name: 'Priya Reddy',
    gender: 'Female',
    age: 25,
    email: 'priya.reddy@example.com',
    phone: '9876543211',
    city: 'Hyderabad',
    state: 'Telangana',
    membership: 'Silver',
    join_date: '2025-02-10',
    is_active: true
  }
]


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({city:'Hyderabad'}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}


# Another Equality Query

We can filter documents using any field present in the document.

The following example retrieves all customers having a **Gold** membership.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE membership = 'Gold';
```

In [8]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"membership": "Gold"})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id': 1006,
 'email': 'ananya.das@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-01-28',
 'membership': 'Gold',
 'name': 'Ananya Das',
 'phone': '9876543215',
 'state': 'West Bengal'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009,

In [9]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({membership:'Gold'}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,
  email: 'ananya.das@example.com',
  phone: '9876543215',
  city: 'Kolkata',
  state: 'West Bengal',
  membership: 'Gold',
  join_date: '2025-01-28',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  emai

# Comparison Query Operators

Comparison operators are used to retrieve documents based on numerical or alphabetical comparisons.

These operators are commonly used for filtering data such as:

- Customers older than a certain age
- Products with prices above a specific value
- Employees with salaries below a threshold

The following comparison operators are supported by MongoDB.

| Operator | Description | SQL Equivalent |
|----------|-------------|----------------|
| `$gt` | Greater Than | `>` |
| `$gte` | Greater Than or Equal To | `>=` |
| `$lt` | Less Than | `<` |
| `$lte` | Less Than or Equal To | `<=` |
| `$ne` | Not Equal To | `!=` |

# Greater Than (`$gt`)

The **`$gt`** operator retrieves documents where the field value is **greater than** the specified value.

The following example retrieves customers whose age is greater than **30**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE age > 30;
```

In [10]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"age": {"$gt": 30}})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_id': 1007,
 'email': 'rohit.verma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-04-01',
 'membership': 'Bronze',
 'name': 'Rohit Verma',
 'phone': '9876543216',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009

In [11]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({age:{$gt:30}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age: 31,
  email: 'rohit.verma@example.com',
  phone: '9876543216',
  city: 'Pune',
  state: 'Maharashtra',
  membership: 'Bronze',
  join_date: '2025-04-01',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  ema

# Greater Than or Equal To (`$gte`)

The **`$gte`** operator retrieves documents where the field value is **greater than or equal to** the specified value.

The following example retrieves customers whose age is **30 years or older**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE age >= 30;
```

In [12]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"age": {"$gte": 30}})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_id': 1007,
 'email': 'rohit.verma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-04-01',
 'membership': 'Bronze',
 'name': 'Rohit Verma',
 'phone': '9876543216',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009

In [13]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({age:{$gte:30}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age: 31,
  email: 'rohit.verma@example.com',
  phone: '9876543216',
  city: 'Pune',
  state: 'Maharashtra',
  membership: 'Bronze',
  join_date: '2025-04-01',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  ema

# Less Than (`$lt`)

The **`$lt`** operator retrieves documents where the field value is **less than** the specified value.

The following example retrieves customers whose age is less than **30**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE age < 30;
```

In [14]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"age": {"$lt": 30}})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'phone': '9876543213',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_

In [15]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({age:{$lt:30}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  email: 'sneha.patel@example.com',
  phone: '9876543213',
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age:

# Less Than or Equal To (`$lte`)

The **`$lte`** operator retrieves documents where the field value is **less than or equal to** the specified value.

The following example retrieves customers whose age is **30 years or younger**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE age <= 30;
```

In [16]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"age": {"$lte": 30}})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'phone': '9876543213',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_

In [17]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({age:{$lte:30}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  email: 'sneha.patel@example.com',
  phone: '9876543213',
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age:

# Not Equal To (`$ne`)

The **`$ne`** operator retrieves documents where the field value is **not equal to** the specified value.

The following example retrieves customers whose membership is **not Gold**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE membership != 'Gold';
```

In [18]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"membership": {"$ne": "Gold"}})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'phone': '9876543213',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_i

In [19]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({membership:{$ne:'Gold'}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  email: 'sneha.patel@example.com',
  phone: '9876543213',
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age

# Logical Query Operators

Logical operators combine multiple query conditions into a single query.

They are useful when data must satisfy one or more conditions simultaneously.

MongoDB provides the following logical operators:

| Operator | Description | SQL Equivalent |
|----------|-------------|----------------|
| `$and` | All conditions must be true | `AND` |
| `$or` | At least one condition must be true | `OR` |
| `$not` | Negates a condition | `NOT` |
| `$nor` | None of the conditions should be true | `NOT (A OR B)` |

# AND Operator (`$and`)

The **`$and`** operator returns documents only when **all specified conditions are true**.

The following example retrieves customers who:

- belong to **Gold** membership, and
- are older than **30** years.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE membership = 'Gold'
AND age > 30;
```

In [20]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "$and": [
        {"membership": "Gold"},
        {"age": {"$gt": 30}}
    ]
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009,
 'email': 'amit.joshi@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-10-12',
 'membership': 'Gold',
 'name': 'Amit Joshi',
 'phone': '9876543218',
 'state': 'Rajasthan'}


In [23]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({$and:[{membership:'Gold'},{age:{$gt:30}}]}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  email: 'amit.joshi@example.com',
  phone: '9876543218',
  city: 'Jaipur',
  state: 'Rajasthan',
  membership: 'Gold',
  join_date: '2024-10-12',
  is_active: true
}


# OR Operator (`$or`)

The **`$or`** operator returns documents when **at least one** of the specified conditions is true.

The following example retrieves customers who are from:

- Hyderabad, or
- Chennai.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE city = 'Hyderabad'
OR city = 'Chennai';
```

In [24]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "$or": [
        {"city": "Hyderabad"},
        {"city": "Chennai"}
    ]
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b91'),
 'age': 40,
 'city': 'Chennai',
 'customer_i

In [26]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({$or:[{city:'Hyderabad'},{city:'Chennai'}]}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b91'),
  customer_id: 1011,
  name: 'Suresh Iyer',
  gender: 'Male',
  age: 4

# NOT Operator (`$not`)

The **`$not`** operator negates a query condition.

The following example retrieves customers whose age is **not greater than 30**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE NOT(age > 30);
```

In [27]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "age": {
        "$not": {
            "$gt": 30
        }
    }
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'phone': '9876543213',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_

In [29]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({age:{$not:{$gt:30}}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  email: 'sneha.patel@example.com',
  phone: '9876543213',
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age:

# NOR Operator (`$nor`)

The **`$nor`** operator returns documents only when **none** of the specified conditions are true.

The following example retrieves customers who are:

- not from Hyderabad, and
- not from Chennai.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE city <> 'Hyderabad'
AND city <> 'Chennai';
```

In [30]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "$nor": [
        {"city": "Hyderabad"},
        {"city": "Chennai"}
    ]
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'phone': '9876543213',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id': 1006,
 'email': 'ananya.das@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-01-28',
 'membership': 'Gold',
 'name': 'Ananya Das',
 'phone': '9876543215',
 'state': 'West Bengal'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_id': 1007

In [32]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({$nor:[{city:'Hyderabad'},{city:'Chennai'}]}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  email: 'sneha.patel@example.com',
  phone: '9876543213',
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,
  email: 'ananya.das@example.com',
  phone: '9876543215',
  city: 'Kolkata',
  state: 'West Bengal',
  membership: 'Gold',
  join_date: '2025-01-28',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age: 31,
  

# Element and Evaluation Query Operators

Element and Evaluation operators allow us to query documents based on:

- Whether a field exists
- The data type of a field
- Whether a field belongs to a list of values
- Whether a field does not belong to a list of values
- Whether a field matches a text pattern

These operators are widely used in real-world applications for searching and filtering data.

# Exists Operator (`$exists`)

The **`$exists`** operator checks whether a particular field exists in a document.

It does **not** check the field's value—it only verifies the presence or absence of the field.

The following example retrieves customers that contain the **phone** field.

In [33]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "phone": {
        "$exists": True
    }
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1

In [35]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({phone:{$exists:true}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,

# In Operator (`$in`)

The **`$in`** operator retrieves documents where a field matches **any one** of the values specified in a list.

The following example retrieves customers whose membership is either:

- Gold
- Platinum

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE membership IN ('Gold', 'Platinum');
```

In [36]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "membership": {
        "$in": ["Gold", "Platinum"]
    }
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'phone': '9876543213',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id':

In [37]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({membership:{$in:['Gold','Platinum']}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  email: 'sneha.patel@example.com',
  phone: '9876543213',
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,

# Not In Operator (`$nin`)

The **`$nin`** operator retrieves documents whose field value is **not present** in the specified list.

The following example retrieves customers whose membership is neither **Gold** nor **Platinum**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE membership NOT IN ('Gold', 'Platinum');
```

In [38]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "membership": {
        "$nin": ["Gold", "Platinum"]
    }
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_id': 1007,
 'email': 'rohit.verma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-04-01',
 'membership': 'Bronze',
 'name': 'Rohit Verma',
 'phone': '9876543216',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8e'),
 'age': 26,
 'city': 'Kochi',
 'customer_id': 1

In [39]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({membership:{$nin:['Gold','Platinum']}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age: 31,
  email: 'rohit.verma@example.com',
  phone: '9876543216',
  city: 'Pune',
  state: 'Maharashtra',
  membership: 'Bronze',
  join_date: '2025-04-01',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8e'),
  customer_id: 1008,
  name: 'Kavya Nair',
  gender: 'Female',
  age: 26,

# Regular Expression Operator (`$regex`)

The **`$regex`** operator performs pattern matching on string fields.

It is useful for searching names, email addresses, cities, and other text-based fields.

The following example retrieves customers whose names start with the letter **A**.

Equivalent SQL query (conceptually):

```sql
SELECT *
FROM customers
WHERE name LIKE 'A%';
```

In [40]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "name": {
        "$regex": "^A"
    }
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id': 1006,
 'email': 'ananya.das@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-01-28',
 'membership': 'Gold',
 'name': 'Ananya Das',
 'phone': '9876543215',
 'state': 'West Bengal'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009,
 'email': 'amit.joshi@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-10-12',
 'membership': 'Gold',
 'name': 'Amit Joshi',
 'phone': '9876543218',
 'state': 'Rajasthan'}


In [42]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({name:{$regex:'^A'}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,
  email: 'ananya.das@example.com',
  phone: '9876543215',
  city: 'Kolkata',
  state: 'West Bengal',
  membership: 'Gold',
  join_date: '2025-01-28',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  email: 'amit.joshi@example.com',
  phone: '9876543218',
  city: 'Jaipur',
  state: 'Rajasthan',
  membership: 'Gold',
  join_date: '2024-10-12',
  is_active: true
}


# Type Operator (`$type`)

The **`$type`** operator retrieves documents where a field is of a specific BSON data type.

This is useful for validating data consistency in a collection.

The following example retrieves documents where the **age** field is stored as an integer.

In [43]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "age": {
        "$type": "int"
    }
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1

In [44]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({age:{$type:'int'}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,

# Projection

By default, MongoDB returns **all fields** of every matching document.

However, in many situations, we need only a few fields instead of the complete document.

**Projection** is used to specify which fields should be included or excluded in the query result.

General Syntax:

```python
collection.find(filter, projection)
```

Equivalent SQL query:

```sql
SELECT column1, column2
FROM table_name;
```

Using projection improves query performance by returning only the required data.

# Include Specific Fields

To include specific fields in the query result, assign the value **1** to those fields in the projection document.

By default, MongoDB also returns the **`_id`** field unless it is explicitly excluded.

The following example displays only the customer's **name**, **city**, and **membership**.

In [45]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find(
    {},
    {
        "name": 1,
        "city": 1,
        "membership": 1
    }
)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'city': 'Hyderabad',
 'membership': 'Gold',
 'name': 'Rahul Sharma'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'city': 'Hyderabad',
 'membership': 'Silver',
 'name': 'Priya Reddy'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'city': 'Bengaluru',
 'membership': 'Gold',
 'name': 'Arjun Kumar'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'city': 'Mumbai',
 'membership': 'Platinum',
 'name': 'Sneha Patel'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'city': 'Chennai',
 'membership': 'Silver',
 'name': 'Vikram Singh'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'city': 'Kolkata',
 'membership': 'Gold',
 'name': 'Ananya Das'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'city': 'Pune',
 'membership': 'Bronze',
 'name': 'Rohit Verma'}
{'_id': ObjectId('6a5cba4483106d9367c89b8e'),
 'city': 'Kochi',
 'membership': 'Silver',
 'name': 'Kavya Nair'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'city': 'Jaipur',
 'membership': 'Gol

In [46]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({}, {name:1, city:1, membership:1}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  name: 'Rahul Sharma',
  city: 'Hyderabad',
  membership: 'Gold'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  name: 'Priya Reddy',
  city: 'Hyderabad',
  membership: 'Silver'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  name: 'Arjun Kumar',
  city: 'Bengaluru',
  membership: 'Gold'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  name: 'Sneha Patel',
  city: 'Mumbai',
  membership: 'Platinum'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  name: 'Vikram Singh',
  city: 'Chennai',
  membership: 'Silver'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  name: 'Ananya Das',
  city: 'Kolkata',
  membership: 'Gold'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  name: 'Rohit Verma',
  city: 'Pune',
  membership: 'Bronze'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8e'),
  name: 'Kavya Nair',
  city: 'Kochi',
  membership: 'Silver'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  name: 'Amit Joshi',
  city: 'Jaipur',
  mem

# Exclude the `_id` Field

The `_id` field is included automatically in every query result.

If it is not required, it can be excluded by assigning **0** to the `_id` field.

The following example displays only the customer's **name**, **city**, and **membership**, without the `_id`.

In [47]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find(
    {},
    {
        "_id": 0,
        "name": 1,
        "city": 1,
        "membership": 1
    }
)

for customer in cursor:
    pprint(customer)

{'city': 'Hyderabad', 'membership': 'Gold', 'name': 'Rahul Sharma'}
{'city': 'Hyderabad', 'membership': 'Silver', 'name': 'Priya Reddy'}
{'city': 'Bengaluru', 'membership': 'Gold', 'name': 'Arjun Kumar'}
{'city': 'Mumbai', 'membership': 'Platinum', 'name': 'Sneha Patel'}
{'city': 'Chennai', 'membership': 'Silver', 'name': 'Vikram Singh'}
{'city': 'Kolkata', 'membership': 'Gold', 'name': 'Ananya Das'}
{'city': 'Pune', 'membership': 'Bronze', 'name': 'Rohit Verma'}
{'city': 'Kochi', 'membership': 'Silver', 'name': 'Kavya Nair'}
{'city': 'Jaipur', 'membership': 'Gold', 'name': 'Amit Joshi'}
{'city': 'Delhi', 'membership': 'Bronze', 'name': 'Neha Sharma'}
{'city': 'Chennai', 'membership': 'Platinum', 'name': 'Suresh Iyer'}


In [48]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({}, {_id:0, name:1, city:1, membership:1}).forEach(doc => printjson(doc))"

{
  name: 'Rahul Sharma',
  city: 'Hyderabad',
  membership: 'Gold'
}
{
  name: 'Priya Reddy',
  city: 'Hyderabad',
  membership: 'Silver'
}
{
  name: 'Arjun Kumar',
  city: 'Bengaluru',
  membership: 'Gold'
}
{
  name: 'Sneha Patel',
  city: 'Mumbai',
  membership: 'Platinum'
}
{
  name: 'Vikram Singh',
  city: 'Chennai',
  membership: 'Silver'
}
{
  name: 'Ananya Das',
  city: 'Kolkata',
  membership: 'Gold'
}
{
  name: 'Rohit Verma',
  city: 'Pune',
  membership: 'Bronze'
}
{
  name: 'Kavya Nair',
  city: 'Kochi',
  membership: 'Silver'
}
{
  name: 'Amit Joshi',
  city: 'Jaipur',
  membership: 'Gold'
}
{
  name: 'Neha Sharma',
  city: 'Delhi',
  membership: 'Bronze'
}
{
  name: 'Suresh Iyer',
  city: 'Chennai',
  membership: 'Platinum'
}


# Exclude Specific Fields

Instead of specifying the required fields, we can exclude unnecessary fields by assigning **0** to them.

> **Important:**  
> MongoDB does **not** allow mixing inclusion (`1`) and exclusion (`0`) in the same projection document, except for the `_id` field.

The following example excludes the **email** and **phone** fields.

In [51]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find(
    {},
    {
        "email": 0,
        "phone": 0
    }
)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d93

In [52]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({}, {email:0, phone:0}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 

# Invalid Projection

MongoDB does **not** allow mixing field inclusion and exclusion in the same projection document (except for `_id`).

For example, the following query is invalid because it attempts to include **name** while excluding **city**.

This query raises an error.

In [53]:
# =====================================
# PyMongo Command
# =====================================

try:
    cursor = customers.find(
        {},
        {
            "name": 1,
            "city": 0
        }
    )

    for customer in cursor:
        pprint(customer)

except Exception as e:
    print(e)

Cannot do exclusion on field city in inclusion projection, full error: {'ok': 0.0, 'errmsg': 'Cannot do exclusion on field city in inclusion projection', 'code': 31254, 'codeName': 'Location31254'}


In [54]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "db.getSiblingDB('ecommerce').customers.find({}, {name:1, city:0})"

# Sorting Query Results

By default, MongoDB returns documents in their natural storage order.

The **`sort()`** method is used to arrange the query results in a specific order.

General Syntax:

```python
collection.find(filter).sort(field, order)
```

where:

- **1** → Ascending Order
- **-1** → Descending Order

Equivalent SQL query:

```sql
SELECT *
FROM customers
ORDER BY age ASC;
```

# Sort in Ascending Order

To sort documents in **ascending order**, specify **1** as the sorting order.

The following example displays all customers sorted by **age** from the youngest to the oldest.

In [55]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find().sort("age", 1)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b90'),
 'age': 24,
 'city': 'Delhi',
 'customer_id': 1010,
 'email': 'neha.sharma@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-05-08',
 'membership': 'Bronze',
 'name': 'Neha Sharma',
 'phone': '9876543219',
 'state': 'Delhi'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b8e'),
 'age': 26,
 'city': 'Kochi',
 'customer_id': 1008,
 'email': 'kavya.nair@example.com',
 'gender': 'Female',
 'is_active': False,
 'join_date': '2025-02-15',
 'membership': 'Silver',
 'name': 'Kavya Nair',
 'phone': '9876543217',
 'state': 'Kerala'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id': 1006,
 'em

In [56]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find().sort({age:1}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b90'),
  customer_id: 1010,
  name: 'Neha Sharma',
  gender: 'Female',
  age: 24,
  email: 'neha.sharma@example.com',
  phone: '9876543219',
  city: 'Delhi',
  state: 'Delhi',
  membership: 'Bronze',
  join_date: '2025-05-08',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8e'),
  customer_id: 1008,
  name: 'Kavya Nair',
  gender: 'Female',
  age: 26,
  email: 'kavya.nair@example.com',
  phone: '9876543217',
  city: 'Kochi',
  state: 'Kerala',
  membership: 'Silver',
  join_date: '2025-02-15',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,
  email: '

# Sort in Descending Order

To sort documents in **descending order**, specify **-1** as the sorting order.

The following example displays customers from the oldest to the youngest.

Equivalent SQL query:

```sql
SELECT *
FROM customers
ORDER BY age DESC;
```

In [57]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find().sort("age", -1)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b91'),
 'age': 40,
 'city': 'Chennai',
 'customer_id': 1011,
 'email': 'suresh.iyer@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-09-25',
 'membership': 'Platinum',
 'name': 'Suresh Iyer',
 'phone': '9876543220',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009,
 'email': 'amit.joshi@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-10-12',
 'membership': 'Gold',
 'name': 'Amit Joshi',
 'phone': '9876543218',
 'state': 'Rajasthan'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 10

In [58]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find().sort({age:-1}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b91'),
  customer_id: 1011,
  name: 'Suresh Iyer',
  gender: 'Male',
  age: 40,
  email: 'suresh.iyer@example.com',
  phone: '9876543220',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Platinum',
  join_date: '2024-09-25',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  email: 'amit.joshi@example.com',
  phone: '9876543218',
  city: 'Jaipur',
  state: 'Rajasthan',
  membership: 'Gold',
  join_date: '2024-10-12',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  ema

# Limit Query Results

The **`limit()`** method restricts the number of documents returned by a query.

This is useful when displaying only a few records, such as:

- Top 5 customers
- First 10 products
- Latest 20 orders

Equivalent SQL query:

```sql
SELECT *
FROM customers
LIMIT 5;
```

In [59]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find().limit(5)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1

In [60]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find().limit(5).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,

# Skip Documents

The **`skip()`** method skips a specified number of documents before returning the results.

It is commonly used with `limit()` to implement **pagination**.

Equivalent SQL concept:

```sql
OFFSET
```

The following example skips the first **3** documents.

In [61]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find().skip(3)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'phone': '9876543213',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id': 1006,
 'email': 'ananya.das@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-01-28',
 'membership': 'Gold',
 'name': 'Ananya Das',
 'phone': '9876543215',
 'state': 'West Bengal'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_id': 

In [62]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find().skip(3).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  email: 'sneha.patel@example.com',
  phone: '9876543213',
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,
  email: 'ananya.das@example.com',
  phone: '9876543215',
  city: 'Kolkata',
  state: 'West Bengal',
  membership: 'Gold',
  join_date: '2025-01-28',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age: 31

# Combining Multiple Operations

MongoDB allows multiple query operations to be chained together.

The following example:

1. Sorts customers by age in descending order.
2. Skips the first two customers.
3. Displays the next three customers.

Equivalent SQL query:

```sql
SELECT *
FROM customers
ORDER BY age DESC
LIMIT 3 OFFSET 2;
```

This technique is widely used in web applications for implementing pagination.

In [63]:
# =====================================
# PyMongo Command
# =====================================

cursor = (
    customers.find()
             .sort("age", -1)
             .skip(2)
             .limit(3)
)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_id': 1007,
 'email': 'rohit.verma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-04-01',
 'membership': 'Bronze',
 'name': 'Rohit Verma',
 'phone': '9876543216',
 'state': 'Maharashtra'}


In [64]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find().sort({age:-1}).skip(2).limit(3).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age: 31,
  email: 'rohit.verma@example.com',
  phone: '9876543216',
  city: 'Pune',
  state: 'Maharashtra',
  membership: 'Bronze',
  join_date: '2025-04-01',
  is_active: true
}
